# 📈 회귀 세션 실습

지난 통계 세션에서는 **"두 변수 사이에 관계가 있는가?"** 를 검정으로 확인해 봤습니다.
이번에는 한 걸음 더 나아가서, 그 관계를 **하나의 식으로 만들어 예측까지** 해 보게 됩니다.

---

- 막힐 때는 **▶ 접힌 힌트**를 펼쳐 보세요!

In [ ]:
# 들어가기 전, 그래프의 한글 깨짐을 방지하는 코드입니다. 실행해 주세요!

import platform
import matplotlib.pyplot as plt

# OS별 한글 폰트 설정
if platform.system() == "Darwin":  # Mac OS
    plt.rc("font", family="AppleGothic")
elif platform.system() == "Windows":  # Windows
    plt.rc("font", family="Malgun Gothic")

# 마이너스(-) 기호 깨짐 방지
plt.rcParams["axes.unicode_minus"] = False

---

# 1부 · 회귀선 만들기


## A. 데이터 탐색하기


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

boston = pd.read_csv('./house_price.csv')   # 파일 경로가 다르면 맞게 수정해 주세요
boston.head()

### 데이터 설명

1978년 보스턴의 주택 가격 자료입니다. **506개 지역**에 대한 기록이 담겨 있어요. 우리는 오늘 이 데이터를 가지고 `주택 가격`을 예측하는 모델을 만들 겁니다!

**종속변수(우리가 맞히고 싶은 것)**

- `MEDV` : 주택 가격 (중앙값, $1000 단위)

**독립변수(예측에 쓸 재료)**

| 칼럼 | 뜻 | | 칼럼 | 뜻 |
| --- | --- | --- | --- | --- |
| `CRIM` | 범죄율 | | `AGE` | 1940년 이전 건축 주택 비율 |
| `ZN` | 대형 거주지역 비율 | | `DIS` | 직업센터까지의 거리 |
| `INDUS` | 비소매상업지역 면적 비율 | | `RAD` | 방사형 고속도로 접근성 |
| `CHAS` | 찰스강 인접 여부 (0/1) | | `TAX` | 재산세율 |
| `NOX` | 일산화질소 농도 | | `PTRATIO` | 학생/교사 비율 |
| `RM` | **주택당 방 수** | | `LSTAT` | 인구 중 하위계층 비율 |

In [ ]:
# info()로 칼럼별 자료형과 결측치를 확인해 볼까요?


### 🤔 그런데 쓰지 않을 칼럼이 두 개 있어요

**`CAT.MEDV`** 는 정답에서 파생된 변수이므로, 이를 다시 독립변수로 쓰면 **답을 보고 답을 맞히는 셈**이 되니 제거해 줍니다.

---

**`B`** 칼럼은 지역 인구 중 흑인 비율을 특정 공식으로 변환한 값입니다.
실제로 이 변수 때문에 **scikit-learn은 2023년(1.2 버전)에 보스턴 데이터셋을 라이브러리에서 완전히 삭제**했어요.
인종 구성이 집값을 설명한다는 전제 자체가 문제이고, 그런 모델이 현실에서 쓰이면 차별을 재생산할 수 있기 때문입니다.

> 💡 **변수를 고르는 일은 통계를 넘어서 다방면의 고려가 필요하다는 점을 염두에 둡시다.**
> 오늘 실습에서는 `B`를 빼고 진행하겠습니다.

In [ ]:
# CAT.MEDV와 B 칼럼을 제거해 봅시다. (drop의 columns 인자를 써보세요)
boston = boston.drop(columns=**)

print('사용할 칼럼:', list(boston.columns))
print('데이터 크기:', boston.shape)

## B. 단순선형회귀 : 하나의 변수를 써 봅시다!


먼저 독립변수를 딱 **하나만** 써서 가장 단순한 형태부터 시작해 봅시다.

$$ y = wx + b $$

`LSTAT`(하위계층 비율)로 `MEDV`(집값)를 설명해 보겠습니다.

#### ① 시각화 : 일단 눈으로 확인!

In [ ]:
# LSTAT를 x축, MEDV를 y축으로 산점도를 그려 봅시다.
plt.figure(figsize=(7, 5))
plt.scatter(**, **, alpha=0.4)
plt.xlabel('LSTAT')
plt.ylabel('MEDV')
plt.title('LSTAT vs MEDV')
plt.show()

하위계층 비율이 높은 지역일수록 집값이 낮아지는 경향이 보이네요.
이 관계를 **직선 하나로** 요약해 봅시다.

#### ② 모델 학습

사이킷런에서는 `LinearRegression()`으로 선형회귀를 할 수 있어요. 학습은 `fit()` 을 이용합니다.

- 모델 생성: `model = LinearRegression()`
- 모델 학습: `model.fit(독립변수, 종속변수)`

> 💡 독립변수를 `[['LSTAT']]` 처럼 대괄호 두 개로 감싸는 이유는?
><BR> -> 사이킷런은 입력을 **2차원(행렬)** 으로 받습니다. 변수가 하나여도 마찬가지예요.
> 반면 정답 `y`는 값 하나씩이니 1차원이어도 괜찮습니다.

In [ ]:
X_simple = boston[['LSTAT']]
y = boston['MEDV']

# 선형회귀 모델을 생성하고, 학습시켜 봅시다.
model = **
model.**(X_simple, y)

print('완료!')

#### ③ 결과 확인 — 모델이 찾아낸 $w$와 $b$

In [ ]:
# 학습된 모델의 회귀계수와 절편을 확인해 봅시다.

print(f'가중치 w (기울기) : {model.coef_[0]:.4f}')
print(f'편향  b (y절편)  : {model.intercept_:.4f}')

**Q.** 나온 값을 세션에서 배운 대로 해석해 봅시다.

- $w$가 음수라는 건 무슨 뜻일까요?
- $b$는 어떤 상황의 값을 나타낼까요?

**A.**

<details>
<summary><b>▶ 힌트</b></summary>
<br>

$w$는 **$x$가 1만큼 늘어날 때 $y$가 얼마나 변하는가**를 의미했습니다.

$b$는 **$x$가 0일 때의 기본값**이었습니다.

</details>

우리는 위에서 `LinearRegression()`으로 회귀 모델을 생성하고, `fit()`으로 모델을 학습시켰습니다.
<br> 이제 이 학습시킨 모델로 예측을 해 봐야겠죠?
<br> 예측은 `predict()`를 사용합니다!

In [ ]:
# predict()로 예측값을 구하고, 회귀선을 데이터 위에 겹쳐 그려 봅시다.
y_pred_simple = model.**(X_simple)

plt.figure(figsize=(7, 5))
plt.scatter(boston['LSTAT'], y, alpha=0.3, label='actual')
plt.plot(boston['LSTAT'], y_pred_simple, color='red', lw=2, label='regression line')
plt.xlabel('LSTAT'); plt.ylabel('MEDV')
plt.legend(); plt.title('Simple Linear Regression')
plt.show()

### 🎯 그런데 왜 하필 이 직선일까요?

세션에서 **잔차**를 배웠죠. 잔차는 실제값과 예측값의 차이였습니다.

$$ 잔차 = y - \hat{y} $$

그리고 최적의 직선을 찾는 방법으로 **최소제곱법(OLS)** 을 배웠습니다.
**잔차를 제곱해서 다 더한 값(RSS)이 가장 작아지는 직선**을 고르는 방법이었어요.

말로만 들으면 와닿지 않으니, 직접 계산해서 확인해 봅시다.

In [ ]:
# 잔차를 구하고, 잔차 제곱합(RSS)을 계산해 봅시다.
residuals_simple = ** - **
rss_model = (residuals_simple ** 2).**()

print(f'잔차 5개만 살펴보기: {residuals_simple[:5].values.round(3)}')
print(f'잔차의 합       : {residuals_simple.sum():.6f}')
print(f'잔차 제곱합(RSS) : {rss_model:,.1f}')

잔차의 **합**은 0에 가깝게 나왔을 거예요. 위아래 오차가 서로 상쇄되기 때문입니다.
그래서 이런 상쇄를 막기 위해, **잔차를 제곱**해야 합니다.

이제 진짜 궁금한 걸 확인해 봅시다. **모델이 찾은 직선이 정말 최선일까요?**
눈대중으로 그은 직선 두 개와 비교해 볼게요.

In [ ]:
def rss_of(w, b):
    pred = w * boston['LSTAT'] + b
    return ((y - pred) ** 2).sum()

candidates = {
    '내가 그은 직선 A (w=-1.0, b=30.0)': rss_of(-1.0, 30.0),
    '내가 그은 직선 B (w=-0.5, b=25.0)': rss_of(-0.5, 25.0),
    f'모델이 찾은 직선  (w={model.coef_[0]:.2f}, b={model.intercept_:.2f})': rss_model,
}

for name, value in candidates.items():
    print(f'{name:45s} RSS = {value:>10,.1f}')

plt.figure(figsize=(7, 5))
plt.scatter(boston['LSTAT'], y, alpha=0.2, color='lightgray')
xs = np.linspace(boston['LSTAT'].min(), boston['LSTAT'].max(), 100)
plt.plot(xs, -1.0 * xs + 30.0, label='A (w=-1.0, b=30)', linestyle=':', lw=2)
plt.plot(xs, -0.5 * xs + 25.0, label='B (w=-0.5, b=25)', linestyle='--', lw=2)
plt.plot(xs, model.coef_[0] * xs + model.intercept_, color='red', label='model', lw=2)
plt.xlabel('LSTAT'); plt.ylabel('MEDV'); plt.legend()
plt.title('어느 직선이 가장 잘 맞을까요?')
plt.show()

**Q.** 세 직선의 RSS를 비교해 봅시다. 무엇을 알 수 있나요?

**A.**

> 🔎 **잠깐, 그런데**
>
> 위 그래프를 다시 보면, 데이터가 살짝 **휘어 있는** 것 같지 않나요?
> 왼쪽 아래로 갈수록 점들이 직선 위로 튀어 오르는 모양이거든요.
>
> 직선으로는 이 곡선을 따라갈 수 없는데…? **이 문제는 이어지는 실습에서 다시 다뤄 보겠습니다!**

## C. 다중선형회귀 : 변수를 여러 개 써봅시다!



$$ y = w_1 x_1 + w_2 x_2 + \cdots + w_n x_n + b $$

`LSTAT` 하나로는 아쉬우니, 이번엔 **가진 변수를 전부** 넣어 보겠습니다.

In [ ]:
# MEDV를 종속변수(y)로, 나머지를 전부 독립변수(X)로 나눠 봅시다.
X = boston.drop(columns=**)
y = boston[**]

print('독립변수 개수:', X.shape[1])
print(list(X.columns))

### Train / Test 나누기

세션 앞부분에서 배운 내용이죠. 모델을 학습시킬 데이터와 검증할 데이터를 나눠 둡니다.

**왜 나눌까요?** 공부한 것과 똑같은 문제로 시험을 보면 실력을 알 수 없는 것처럼,
학습에 쓰지 않은 데이터로 평가해야 **처음 보는 데이터에도 통하는지** 알 수 있어요.

`test_size`는 테스트 사이즈의 크기를 결정합니다. 즉, `test_size=0.2`는 전체의 20%를 테스트용으로 떼어 둔다는 뜻이에요.

In [ ]:
# 전체 데이터의 20%를 테스트용으로 사용합시다!

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=**, random_state=42
)

print(f'train: {X_train.shape[0]}개  /  test: {X_test.shape[0]}개')

### statsmodels와 sklearn, 두 가지 도구

같은 선형회귀인데 파이썬에는 도구가 두 개 있습니다. 성격이 조금 달라요.

| | **statsmodels** | **scikit-learn** |
| --- | --- | --- |
| 성격 | 통계 분석 도구 | 머신러닝 도구 |
| 강점 | p-value, F통계량 등 **통계 장표가 풍부** | 예측·평가 기능이 다양하고 간편 |
| 주 용도 | "이 변수가 유의한가?" **설명** | "얼마나 잘 맞히나?" **예측** |
| 절편 | `add_constant()`로 **직접 추가** | 기본으로 포함 |

> 💡 오늘 세션자료 3️⃣장에서 다뤘던 F-statistic, Adjusted R², AIC/BIC는 **statsmodels에서만** 볼 수 있어요.
> 그래서 오늘은 statsmodels를 중심으로 쓰고, 예측 성능 평가는 sklearn을 함께 씁니다.

In [ ]:
# sklearn으로 다중회귀 모델을 만들어 봅시다.
lr = LinearRegression()
lr.**(X_train, y_train)              # 학습
y_pred_skl = lr.**(X_test)           # 예측

print(f'sklearn 예측값 5개: {y_pred_skl[:5].round(2)}')
print(f'테스트 R2: {r2_score(y_test, y_pred_skl):.4f}')

이번엔 statsmodels로 같은 모델을 만들어 봅시다.
statsmodels의 `OLS`는 **절편이 기본으로 없어서** `add_constant()`로 직접 넣어줘야 합니다.

- 모델 선언: `sm.OLS(종속변수, 독립변수)` ← **순서에 주의하세요!**
- 모델 학습: `model.fit()`

In [ ]:
X_train_c = sm.add_constant(X_train)
X_test_c = sm.add_constant(X_test)

# statsmodels의 OLS로 모델을 만들고 학습시켜 봅시다. (종속변수가 먼저!)
ols_full = sm.OLS(**, **).**()

print(f'R-squared      : {ols_full.rsquared:.4f}')
print(f'Adj. R-squared : {ols_full.rsquared_adj:.4f}')

`LSTAT` 하나만 썼을 때 R²가 약 0.54였는데, 변수를 다 넣으니 **0.74 근처**까지 올라왔네요.
변수를 여러 개 쓰는 게 확실히 도움이 됩니다.

그런데… **변수가 많으면 무조건 좋은 걸까요?** 아래에서 확인해 봅시다.

## D. 다중공선성 : 변수들끼리 겹치지 않았나요?

세션에서 이런 예시를 봤었죠. **'주택의 크기'와 '방의 개수'** 를 둘 다 넣으면,
둘이 거의 같은 이야기를 하고 있어서 **어느 쪽이 집값에 영향을 주는지 구분할 수 없게** 된다고요.

지금 우리 모델에는 변수가 12개나 있습니다. 겹치는 변수가 없는지 확인해 봅시다.

### D-1. 상관계수로 훑어보기

통계 세션에서 쓰던 히트맵이 등장합니다. 반갑죠? 🙂

In [ ]:
# 독립변수들끼리의 "상관계수"를 히트맵으로 그려 봅시다.
plt.figure(figsize=(12, 9))
sns.heatmap(X.**(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('독립변수 간 상관계수')
plt.show()

**Q.** 유난히 새빨간(또는 새파란) 칸이 보이시나요? **상관계수 절댓값이 0.7을 넘는 쌍**을 찾아보세요.

**A.**



### D-2. VIF로 수치화하기

상관계수는 **두 변수씩만** 볼 수 있다는 한계가 있어요.
세 변수가 조금씩 겹치는 경우는 잡아내지 못합니다. 그래서 쓰는 것이 **VIF(분산팽창인수)** 입니다.

세션에서 배웠던 판단 기준은 다음과 같았습니다.

| VIF | 판단 |
| --- | --- |
| 1 | 다른 변수와 전혀 상관없음 |
| 5 미만 | 문제없음 |
| 5 ~ 10 | 주의 |
| 10 초과 | 심각 |

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def vif_wrong(df_X):
    return pd.Series(
        [variance_inflation_factor(df_X.values, i) for i in range(df_X.shape[1])],
        index=df_X.columns
    ).sort_values(ascending=False)

print(vif_wrong(X).round(2).to_string())

### ⚠️ 잠깐, 이 결과 좀 이상하지 않나요?

`PTRATIO`(학생/교사 비율)의 VIF가 **78**로 나왔습니다. 심각 기준인 10의 여덟 배죠.
그런데 방금 히트맵을 다시 보세요. **`PTRATIO`와 강하게 상관된 변수가 있었나요?**

없었습니다. 가장 높은 게 `INDUS`와 0.38 정도였어요.
상관관계가 그렇게 약한 변수의 VIF가 78이 나올 수는 없습니다. **계산에 문제가 있다는 뜻이에요.**

---

**원인은 절편(상수항)입니다.**

VIF는 내부적으로 *"이 변수를 나머지 변수들로 예측했을 때 얼마나 잘 맞는가"* 를 회귀로 계산해요.
그런데 이때 **절편이 없으면** 그 회귀가 원점을 억지로 지나가야 해서 R²가 엉뚱하게 부풀려집니다.

`PTRATIO`는 값이 대체로 15~22 사이라 0에서 멀리 떨어져 있습니다. 그래서 특히 크게 왜곡된 거예요.

**해결책은 간단합니다. `add_constant()`로 절편을 넣고 계산하면 됩니다.**

In [ ]:
def vif_of(df_X):
    Xc = sm.**(df_X)                     # ← 절편을 추가하는 함수는?
    s = pd.Series(
        [variance_inflation_factor(Xc.values, i) for i in range(Xc.shape[1])],
        index=Xc.columns
    )
    return s.drop('const').sort_values(ascending=False)

compare = pd.DataFrame({
    '절편 없이 (X)': vif_wrong(X).round(2),
    '절편 포함 (O)': vif_of(X).round(2),
})
display(compare)

**Q.** 두 열의 값이 얼마나 달라졌나요? 절편을 넣고 계산했을 때 **가장 VIF가 높은 변수 두 개**는 무엇인가요?

**A.**

<details>
<summary><b>▶ 정리가 잘 안 된다면</b></summary>
<br>

`PTRATIO`, `NOX`, `RM` 같은 변수들의 값이 어떻게 변했는지 먼저 보세요.

그리고 히트맵에서 상관계수 0.91로 가장 높았던 그 쌍을 떠올려 보시면 됩니다.

</details>

### D-3. 변수 하나 줄여보기

위 결과를 보면, `TAX`(9.00)와 `RAD`(7.45)가 "주의" 구간에 있네요.

세션자료의 대처 방법 중 가장 간단한 **변수 제거**를 해봅시다.
둘 중 하나만 빼면 나머지 하나도 함께 내려갈 거예요.

> 💡 **하나 지우면 다른 변수들의 VIF도 전부 바뀝니다.**
> 그래서 한꺼번에 여러 개를 지우기보다 **하나씩 지우면서 다시 확인**하는 게 좋아요.

In [ ]:
# VIF가 가장 높은 TAX를 빼고 다시 계산해 봅시다.
print(vif_of(X.drop(columns=**)).round(2).to_string())

**Q.** `TAX` 하나를 뺐을 뿐인데 `RAD`는 어떻게 되었나요?

**A.**

<details>
<summary><b>▶ 이유는?</b></summary>
<br>

- `RAD`가 7.45 → **약 2.77** 로 뚝 떨어집니다. 이제 전체 변수 중 최대 VIF가 `NOX`의 **4.34** 예요.
- `RAD`의 VIF가 높았던 이유가 **오로지 `TAX` 때문**이었다는 뜻이죠. 짝이 사라지니 문제도 사라진 겁니다.
- 이렇기 때문에 변수를 **하나씩** 지우면서 확인해야 합니다. 한 번에 둘 다 지웠다면 멀쩡한 `RAD`까지 잃을 수도 있으니까요.

</details>

In [ ]:
X = X.drop(columns='TAX')
X_train = X_train.drop(columns='TAX')
X_test = X_test.drop(columns='TAX')

print('남은 독립변수:', list(X.columns))

## E. 규제선형모델 맛보기 : 망치로 눌러보기


세션에서 릿지와 라쏘를 **망치**에 비유했었죠.

> 🔨 회귀계수 기둥들이 너무 높이 솟으면(= 특정 변수에 과하게 의존하면) 과적합이 생기니, 망치로 눌러 낮춰준다.

말로만 들으면 감이 잘 안 오니, **실제로 눌러서 낮아지는지** 눈으로 확인해 봅시다.

`alpha`가 규제의 세기예요. 크게 할수록 세게 누르는 셈이죠.

> 💡 규제를 쓸 땐 **스케일링이 꼭 필요합니다.** 변수마다 단위가 제각각이면(재산세율은 수백, 방 개수는 한 자리)
> 큰 단위 변수만 억울하게 세게 얻어맞거든요. 모두에게 공평한 망치가 되려면 단위를 먼저 맞춰줘야 해요.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

rows = []
for a in [0.01, 1, 100, 1000]:
    ridge = Ridge(alpha=**).fit(X_train_scaled, y_train)      # alpha에 무엇을 넣어야 할까요?
    rows.append({
        'alpha': a,
        '계수 절댓값 합': round(np.abs(ridge.coef_).sum(), 3),
        '가장 큰 계수': round(np.abs(ridge.coef_).max(), 3),
    })

display(pd.DataFrame(rows))

**Q.** `alpha`가 커질수록 회귀계수는 어떻게 변하나요? 이게 세션의 망치 비유와 어떻게 연결되나요?

**A.**

<details>
<summary><b>▶ 한 가지 더 생각해 볼 거리</b></summary>
<br>

- `alpha`를 무한정 크게 하면 계수가 결국 0에 가까워질 텐데, 그럼 모델은 어떤 상태가 될까요?
- 세션자료에서 **과소적합**이라고 불렀던 상태가 됩니다.
- 결국 `alpha`는 **"오차를 줄이는 것"과 "특정 변수에 집착하지 않는 것" 사이의 균형점**을 정하는 손잡이인 셈이에요.

> 💡 릿지(L2)는 계수를 **작게** 만들 뿐 0으로 만들지는 않아요. 0으로 만들어 변수를 아예 없애는 건 라쏘(L1)였죠.
> 오늘은 여기까지만 확인하고 넘어갑니다!

</details>

---

# ☕ 여기까지가 실습 1부입니다

**1부에서 한 것**

- 직선 하나로 시작해서 **잔차와 RSS**로 "왜 이 직선인가"를 확인했어요 (최소제곱법)
- 변수를 전부 넣어 다중회귀를 만들고, R²를 0.54 → 0.74로 끌어올렸습니다
- **VIF의 절편 함정**을 피해 가며 겹치는 변수(`TAX`)를 정리했어요
- 규제가 회귀계수를 실제로 눌러 낮추는 것을 확인했습니다

잠시 쉬고 세션 후반부에서 만나요! 🙂

---

# 2부 · 모델 따져보기


1부에서 만든 변수들을 그대로 이어서 씁니다. **1부 셀을 먼저 실행한 상태여야 해요!**

## F. 통계 장표 읽기

statsmodels의 `summary()`는 모델에 대한 정보를 한 장에 몰아 요약해서 보여줍니다.
처음 보면 숫자가 너무 많아 당황스럽지만, **볼 곳이 몇 군데 정해져 있어요.**

In [ ]:
X_train_c = sm.add_constant(X_train)
X_test_c = sm.add_constant(X_test)

# OLS 모델을 학습시키고, 통계 장표를 출력해 봅시다.
ols = sm.OLS(y_train, X_train_c).fit()
print(ols.**())

### 어디를 봐야 할까요?

**① 모델 전체가 유의한가?**

| 항목 | 보는 법 |
| --- | --- |
| `F-statistic` | 클수록 모델이 유의미. *"모델이 설명하는 변동 ÷ 설명 못 하는 변동"* |
| `Prob (F-statistic)` | 0.05보다 작으면 **적어도 하나의 변수는** 집값과 관계가 있다 |

**② 모델이 얼마나 설명하는가?**

| 항목 | 보는 법 |
| --- | --- |
| `R-squared` | 0~1. 1에 가까울수록 잘 설명. 단 **변수를 넣기만 해도 올라감** |
| `Adj. R-squared` | 변수 개수에 **벌점**을 준 R². 변수를 늘릴 땐 이쪽을 봐야 해요 |
| `AIC` / `BIC` | **작을수록** 좋음. 복잡도에 벌점을 주는 지표로, BIC가 더 엄격합니다 |



**③ 개별 변수가 유의한가?**

| 항목 | 보는 법 |
| --- | --- |
| `coef` | 그 변수의 회귀계수 $w$ |
| `P > t (절대값) ` | 0.05보다 작으면 **이 변수는 유의미하다** |

**④ 아래쪽 진단 지표** : `Durbin-Watson`, `Jarque-Bera`, `Omnibus`
→ 가정을 검정할 때 그대로 씁니다. **이미 계산되어 나와 있어요!**

**Q.** 장표를 보고 세 가지를 확인해 보세요.

1. 이 모델은 통계적으로 유의한가요? (`Prob (F-statistic)`)
2. `R-squared`와 `Adj. R-squared`의 차이는 얼마나 되나요?
3. `P>|t|`가 0.05를 넘는 변수는 무엇인가요?

**A.**

<details>
<summary><b>▶ 한 가지 더 생각해 볼 거리</b></summary>
<br>

> 💡 여기서 쓰는 t-검정, 눈에 익지 않나요? **지난 통계 세션에서 배운 그 t-검정이 맞습니다.**
> 그때는 두 집단의 평균을 비교했고, 지금은 **회귀계수가 0인지**를 확인하고 있어요. 원리는 똑같습니다.

</details>

### 유의하지 않은 변수를 빼면 좋아질까요?

`AGE`의 p-value가 0.82로 가장 높네요. **한 번에 네 개를 다 지우기보다 하나씩** 빼보겠습니다.
(VIF 때와 같은 이유예요. 하나를 빼면 나머지 변수들의 p-value도 전부 바뀌기 때문입니다.)

In [ ]:
cur = list(X_train.columns)
log = []

for _ in range(6):
    m = sm.OLS(y_train, sm.add_constant(X_train[cur])).fit()
    pv = m.pvalues.drop('const').sort_values(ascending=False)
    log.append({
        '변수 개수': len(cur),
        'R2': round(m.rsquared, 4),
        'Adj R2': round(m.rsquared_adj, 4),
        'AIC': round(m.aic, 1),
        'BIC': round(m.bic, 1),
        '최대 p-value 변수': f'{pv.index[0]} ({pv.iloc[0]:.3f})',
    })
    if pv.iloc[0] < 0.05:
        break
    cur.remove(pv.index[0])

display(pd.DataFrame(log))
print('\n모든 변수가 유의해진 시점의 변수들:', cur)

**Q.** 표를 보면 재미있는 일이 벌어집니다.

- `R2`는 변수를 뺄 때마다 어떻게 되나요?
- `Adj R2`, `AIC`, `BIC`가 **각각 가장 좋은 지점**은 어디인가요? 셋이 일치하나요?

**A.**

<details>
<summary><b>▶ 표 읽는 법</b></summary>
<br>

`R2`는 **클수록**, `Adj R2`도 **클수록** 좋아요.

반면 `AIC`와 `BIC`는 **작을수록** 좋습니다. 부호가 반대라는 점에 주의하세요!

</details>

In [ ]:
final_cols = cur
X_train_f = sm.add_constant(X_train[final_cols])
X_test_f = sm.add_constant(X_test[final_cols])

final_model = sm.OLS(y_train, X_train_f).fit()
print('최종 변수:', final_cols)
print(f'Adj R2 = {final_model.rsquared_adj:.4f}')

## G. 회귀 가정 6가지 검정하기


세션에서 배웠듯, 선형회귀를 믿고 쓰려면 만족해야 할 가정들이 있어요.
그런데 이 가정들은 **모델을 만들기 전에는 확인할 수 없는 것**이 많았죠. 오차항이 없으니까요.

| 가정 | 분석 전 | 분석 후 | 확인 방법 |
| --- | --- | --- | --- |
| 1. 선형성 | ✅ | ✅ | 산점도, 잔차 도표 |
| 2. 독립성(다중공선성) | ✅ | ✅ | VIF, 상관계수 |
| 3. 오차항 평균 = 0 | ❌ | ✅ | `np.mean(residuals)` |
| 4. 등분산성 | ❌ | ✅ | 잔차 도표 |
| 5. 자기상관 없음 | 🔺 | ✅ | Durbin-Watson |
| 6. 정규성 | ❌ | ✅ | Shapiro-Wilk, Q-Q plot |

**이제 우리는 모델이 있으니**, 여섯 가지를 전부 확인해 볼 수 있습니다.
먼저 잔차부터 확인할게요.

In [ ]:
# 학습된 모델에서 잔차와 예측값을 꺼내 봅시다.
# 힌트: final_model.resid , final_model.fittedvalues
residuals = **
fitted = **

print(f'잔차 개수: {len(residuals)}')
print(f'잔차 5개: {residuals[:5].values.round(3)}')

### 가정 1 · 선형성

관계가 정말 직선 모양인지 봅니다. **잔차를 예측값에 대해 찍어보면** 알 수 있어요.
잔차가 0 주변에 **아무 패턴 없이 흩어져 있으면** 선형성이 잘 지켜진 겁니다.

In [ ]:
# 예측값(x축)과 잔차(y축)의 산점도를 그려 봅시다.
plt.figure(figsize=(8, 5))
plt.scatter(**, **, alpha=0.5)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('예측값 (fitted)')
plt.ylabel('잔차 (residuals)')
plt.title('Residuals vs Fitted')
plt.show()

**Q.** 잔차가 정말 아무 패턴 없이 흩어져 있나요? 혹시 **곡선 모양**이 보이지는 않나요?

**A.**

<details>
<summary><b>▶ 열어 보세요</b></summary>
<br>

- 완전히 무작위로 보이지는 않습니다. 잔차가 전체적으로 **아래로 볼록한 U자 곡선** 형태를 그리고 있어요.
- 예측값이 낮은 구간과 높은 구간에서 잔차가 양수로 커지고, 중간 구간에서는 음수 쪽으로 몰립니다.
- 이건 **직선만으로는 담아내지 못한 곡선 관계가 남아 있다**는 신호예요. 즉 **선형성 가정이 완벽하게는 지켜지지 않습니다.**
- 오른쪽 위에 일직선으로 늘어선 점들도 보일 거예요. `MEDV`가 50에서 잘려 있어서(상한 처리) 생기는 현상입니다.

> 🔎 앞선 실습에서 "데이터가 휘어 있는 것 같다"고 했던 것 기억나시나요? **여기서도 같은 이야기가 나오고 있어요.**

</details>

### 가정 2 · 독립성 (다중공선성)

이건 이미 확인했었죠! VIF로 확인하고 `TAX`까지 제거했으니 넘어가도 되지만,
**변수를 더 뺐으니** 최종 모델 기준으로 한 번만 다시 확인해 볼게요.

In [ ]:
# 최종 변수들의 VIF를 다시 확인해 봅시다. (1부에서 만든 vif_of 함수를 쓰면 됩니다)
print(**(X_train[final_cols]).round(2).to_string())

### 가정 3 · 오차항의 평균은 0

잔차의 평균이 0에 가까워야 합니다. 그렇지 않다면 모델이 **항상 일정하게 빗나가고 있다**는 뜻이거든요.

In [ ]:
# 잔차의 평균을 구해 봅시다.
print(f'잔차의 평균: {np.**(residuals):.10f}')
print(f'지수 표기   : {np.mean(residuals):.3e}')

거의 0이 나왔죠? `e-14` 같은 표기는 소수점 아래 14자리라는 뜻이니 사실상 0입니다.

> 💡 사실 이 가정은 **절편이 있는 최소제곱법에서는 항상 성립**해요. OLS가 그렇게 계산하도록 설계되어 있습니다.
> 그래서 이 가정이 깨졌다면 보통 **절편을 빼먹었다**는 신호랍니다.

### 가정 4 · 등분산성

잔차의 **흩어진 정도**가 구간마다 일정해야 한다는 가정이에요.
가정 1에서 그린 그래프를 이번엔 **세로 폭**에 집중해서 다시 봅시다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].scatter(fitted, residuals, alpha=0.5)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_title('Residuals vs Fitted')
axes[0].set_xlabel('fitted'); axes[0].set_ylabel('residuals')

axes[1].scatter(fitted, np.sqrt(np.abs(residuals)), alpha=0.5)
axes[1].set_title('Scale-Location (퍼짐 정도만 보기)')
axes[1].set_xlabel('fitted'); axes[1].set_ylabel('sqrt(|residuals|)')

plt.tight_layout()
plt.show()

**Q.** 잔차의 세로 폭이 왼쪽부터 오른쪽까지 **일정한가요?** 등분산성이 지켜졌다고 볼 수 있을까요?

**A.**

<details>
<summary><b>▶ 열어 보세요. </b></summary>
<br>

- **일정하지 않습니다.** 예측값이 작은 구간에서는 잔차가 비교적 좁게 모여 있는데, 예측값이 커질수록 위아래로 **점점 넓게 퍼집니다.**
- 오른쪽 Scale-Location 그래프에서도 선이 평평하지 않고 우상향하는 경향이 보여요.
- 즉 **이분산성(heteroscedasticity)** 이 존재합니다. 세션자료 그림 1에 가까운 상황이에요.
- 이 말은 **비싼 집일수록 모델의 예측이 부정확해진다**는 뜻입니다. 싼 집은 잘 맞히는데 비싼 집은 잘 못 맞히는 거죠.

> 📌 세션자료에 나왔듯 이럴 땐 종속변수에 **로그 변환**을 하거나, 다른 형태의 모델을 고려해 볼 수 있어요.

</details>

### 가정 5 · 오차항의 자기상관 없음

잔차끼리 서로 관련이 있으면 안 된다는 가정이에요. 주로 **시계열 데이터**에서 문제가 됩니다.

**Durbin-Watson** 값으로 확인하는데, 판단 기준은 이랬죠.

- **2에 가까우면** → 자기상관 없음 ✅
- 2보다 많이 작으면 → 양의 자기상관
- 2보다 많이 크면 → 음의 자기상관

> 💡 이 값은 **`summary()` 장표 오른쪽 아래에 이미 찍혀 있어요.** 따로 구할 필요가 없답니다.

In [ ]:
from statsmodels.stats.stattools import durbin_watson

# 잔차의 Durbin-Watson 값을 구해 봅시다.
dw = durbin_watson(**)
print(f'Durbin-Watson: {dw:.4f}')

**Q.** 나온 값을 기준에 비추어 보면 어떻게 판단할 수 있을까요? 그리고 이 데이터에서 **애당초 자기상관을 걱정할 필요가 있었을까요?**

**A.**

<details>
<summary><b>▶ 열어 보세요. </b></summary>
<br>

- DW ≈ **2.16** 으로 2에 아주 가깝습니다. **자기상관은 없다**고 판단할 수 있어요. 이 가정은 통과!
- 사실 예상된 결과이기도 합니다. 이 데이터는 **시계열이 아니라 지역별 횡단면 자료**거든요. 행의 순서에 시간적 의미가 없으니 자기상관이 생길 이유가 적어요.
- 세션자료에서 자기상관은 **시계열 자료에서 많이 나타난다**고 했던 이유가 이것입니다. 주가나 월별 매출처럼 **이전 값이 다음 값에 영향을 주는** 데이터에서 문제가 되죠.

</details>

### 가정 6 · 정규성

오차항이 정규분포를 따르는지 확인합니다. 두 가지 방법을 함께 써볼게요.

- **Shapiro-Wilk 검정** — 수치로 판단
  - H₀: 잔차가 정규분포를 따른다
  - H₁: 잔차가 정규분포를 따르지 않는다
- **Q-Q plot** — 눈으로 판단. 점들이 빨간 대각선 위에 일직선으로 놓이면 정규분포에 가까워요

In [ ]:
from scipy.stats import shapiro

# 잔차에 대해 Shapiro-Wilk 검정을 수행해 봅시다. (반환값은 (통계량, p-value) 튜플이에요)
stat, p_value = shapiro(**)
print(f'Shapiro-Wilk p-value: {p_value:.3e}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(residuals, kde=True, bins=30, ax=axes[0])
axes[0].set_title('잔차의 분포')

sm.qqplot(residuals, line='s', ax=axes[1])
axes[1].set_title('Q-Q Plot')

plt.tight_layout()
plt.show()

**Q.** Shapiro-Wilk 검정 결과와 Q-Q plot이 같은 이야기를 하고 있나요? 정규성 가정은 만족되었나요?

**A.**

<details>
<summary><b>▶ 열어 보세요. </b></summary>
<br>

- p-value가 **약 1.3e-14** 로 0.05보다 훨씬 작습니다. 귀무가설("정규분포를 따른다")을 **기각**해야 해요.
  즉 **정규성 가정은 만족되지 않습니다.**
- Q-Q plot도 같은 이야기를 합니다. 가운데는 대체로 직선을 따라가지만, **양 끝(특히 오른쪽 위)이 위로 크게 휘어** 있어요.
- 히스토그램을 보면 오른쪽으로 긴 꼬리가 있습니다. 모델이 크게 **과소예측**한 지역들이 있다는 뜻이에요.
- 두 방법의 결론이 일치하네요. 세션자료 말대로 **수치 검정과 시각화는 함께 봐야** 합니다. 만능 도구는 없으니까요!

</details>

### 🤔 정리해 봅시다 — 가정 몇 개가 깨졌는데, 이제 어떡하죠?

**Q.** 여섯 가지 중 지켜진 것과 깨진 것을 정리하고, **이 모델을 그대로 써도 될지** 판단해 보세요.

**A.**
| 가정 | 결과 |
| --- | --- |
| 1. 선형성 |  |
| 2. 독립성 |  |
| 3. 오차평균 0 | |
| 4. 등분산성 |  |
| 5. 자기상관 | |
| 6. 정규성 |  |



<details>
<summary><b>▶ 열어 보세요 </b></summary>
<br>

- 세션자료에 나온 대로, 가정을 **100% 만족시키기는 원래 어렵습니다.** 다음 두 가지로 타협할 수 있어요.
- **① 데이터 개수** : 학습 데이터가 404개로 충분히 많습니다(변수 7개 × 10~20개 기준을 넉넉히 넘어요). 중심극한정리 덕분에 **정규성 위배는 어느 정도 눈감아줄 수 있습니다.**
- **② 분석의 목적** :
  - **설명**이 목적이라면(어느 변수가 집값에 영향을 주는가) 지금 상태로는 곤란합니다. 등분산성이 깨지면 **p-value를 그대로 믿기 어렵거든요.**
  - **예측**이 목적이라면 MAE나 RMSE가 납득할 만한 수준인지 보고 그대로 쓸 수도 있어요.
- **가장 근본적인 해결책은 선형성부터 다시 보는 것**입니다. 잔차의 곡선 패턴과 이분산성은 사실 **같은 원인**에서 나올 때가 많거든요. 관계가 직선이 아닌데 직선을 억지로 그은 것이죠.

> 💡 그래서 **다음으로는 비선형 모델을 시도해 볼 겁니다.** 조금만 기다려 주세요!


</details>

## H. 성능 평가 지표


지금까지는 **학습 데이터**로만 모델을 이야기했습니다. 이제 모델이 **한 번도 학습하지 않은 테스트 데이터**로 모델의 진짜 실력을 평가해 봅시다.

| 지표 | 계산 | 성격 |
| --- | --- | --- |
| **MSE** | 오차를 **제곱**해서 평균 | 큰 오차에 민감 |
| **MAE** | 오차의 **절댓값**을 평균 | 모든 오차를 동등하게 |
| **R²** | 평균으로 예측할 때보다 얼마나 나은가 | 0~1, 클수록 좋음 |

In [ ]:
# 테스트 데이터로 예측하고, 세 가지 지표를 구해 봅시다.
y_test_pred = final_model.**(X_test_f)

print(f'Test MSE  : {mean_squared_error(y_test, y_test_pred):.3f}')
print(f'Test RMSE : {np.sqrt(mean_squared_error(y_test, y_test_pred)):.3f}')
print(f'Test MAE  : {**(y_test, y_test_pred):.3f}')
print(f'Test R2   : {**(y_test, y_test_pred):.4f}')

MAE가 약 3.3이 나왔을 거예요. `MEDV`의 단위가 $1000이니 **평균적으로 3,300달러쯤 빗나간다**는 뜻입니다.

> 💡 **MSE는 제곱한 값이라 단위가 달라져서 해석이 어려워요.** 그래서 제곱근을 씌운 **RMSE**를 자주 씁니다.
> RMSE는 MAE처럼 원래 단위로 읽을 수 있으면서, 큰 오차에 민감한 MSE의 성질도 유지하거든요.

### MSE와 MAE, 뭐가 다른가요?

세션자료에서 **MSE는 이상치에 민감하고 MAE는 덜하다**고 했죠. 작은 실험으로 확인해 봅시다.

In [ ]:
y_true = np.array([10., 12., 11., 13., 12.])
y_ok   = np.array([11., 11., 12., 12., 11.])     # 다섯 개 모두 1씩 빗나감

y_bad = y_ok.copy()
y_bad[0] = 30.                                    # 한 개만 크게 빗나감

result = pd.DataFrame({
    '상황': ['모두 조금씩 틀림', '하나만 크게 틀림'],
    'MSE': [mean_squared_error(y_true, y_ok), mean_squared_error(y_true, y_bad)],
    'MAE': [mean_absolute_error(y_true, y_ok), mean_absolute_error(y_true, y_bad)],
}).round(2)

display(result)

**Q.** 한 개만 크게 틀렸을 때 MSE와 MAE는 각각 몇 배가 되었나요? 어느 쪽이 더 크게 반응하나요?
그리고 **집값 예측에서는 어느 지표를 보는 게 좋을까요?**

**A.**

<details>
<summary><b>▶ 열어 보세요 </b></summary>
<br>

- MSE는 **1.00 → 80.80 으로 약 81배**, MAE는 **1.00 → 4.80 으로 약 4.8배** 커졌습니다.
- 똑같이 한 개가 크게 틀렸을 뿐인데 **MSE가 훨씬 격렬하게 반응**합니다. 오차를 제곱하기 때문이에요.
- 어느 쪽을 볼지는 **목적에 따라 다릅니다.**
  - **큰 실수를 절대 하면 안 되는 상황**이라면 MSE(또는 RMSE)를 봐야 해요. 큰 오차에 벌점을 세게 주니까요.
  - **전반적으로 얼마나 맞히는지**가 궁금하고 이상치에 휘둘리기 싫다면 MAE가 낫습니다.
- 집값 예측이라면
  - 고가 주택 몇 채를 크게 틀리는 것이 문제라면 RMSE를,
  - 일반적인 예측 성능을 보고 싶다면 MAE를 보면 됩니다.

> 💡 통계 세션에서 **평균은 이상치에 민감하고 중앙값은 덜하다**고 배웠던 것 기억하시나요? 구조가 완전히 같습니다.
> **제곱하거나 극단값을 그대로 쓰면 이상치에 흔들리고, 절댓값이나 순위를 쓰면 덜 흔들린다고 생각하면 쉽습니다!.**
</details>

## I. 비선형회귀

앞에서 이런 이야기를 했었습니다

> 🔎 *"데이터가 살짝 휘어 있는 것 같지 않나요? 직선으로는 이 곡선을 따라갈 수 없는데…"*

이제 이 문제를 한번 해결해 봅시다. 앞에서 봤던 `LSTAT` - `MEDV` 산점도로 돌아가 봐요!

In [ ]:
X_ls = boston[['LSTAT']].values
y_ls = boston['MEDV'].values

plt.figure(figsize=(7, 5))
plt.scatter(X_ls, y_ls, alpha=0.4, color='lightgray')
plt.xlabel('LSTAT'); plt.ylabel('MEDV')
plt.title('다시 만난 LSTAT vs MEDV')
plt.show()

### I-1. 다항회귀 — 곡선을 그어봅시다

세션자료에서 배운 방법이에요. 원래 식에 **제곱항, 세제곱항을 추가**하는 겁니다.

$$ Y = a + bX \quad \rightarrow \quad Y = a + bX + cX^2 + dX^3 $$

`PolynomialFeatures(degree=n)`을 쓰면 알아서 만들어 줍니다.

> 💡 **재미있는 점**: 항을 추가한 뒤에는 그냥 **평소처럼 선형회귀를 돌리면 됩니다.**
> $X^2$을 그냥 새로운 변수 하나로 취급하는 거예요. 그래서 이름이 여전히 "선형"회귀랍니다!

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# 2차식, 3차식 특성을 만들어 봅시다. degree에 알맞은 숫자를 넣어주세요.
quadratic = PolynomialFeatures(degree=**)
cubic = PolynomialFeatures(degree=**)

X_quad = quadratic.fit_transform(X_ls)
X_cubic = cubic.fit_transform(X_ls)

print(f'원래 shape : {X_ls.shape}')
print(f'2차 변환 후: {X_quad.shape}   (1, x, x^2)')
print(f'3차 변환 후: {X_cubic.shape}   (1, x, x^2, x^3)')

In [ ]:
X_fit = np.arange(X_ls.min(), X_ls.max(), 0.5)[:, np.newaxis]
regr = LinearRegression()

regr.fit(X_ls, y_ls)
y_lin_fit = regr.predict(X_fit)
r2_lin = r2_score(y_ls, regr.predict(X_ls))

regr.fit(X_quad, y_ls)
y_quad_fit = regr.predict(quadratic.fit_transform(X_fit))
r2_quad = r2_score(y_ls, regr.predict(X_quad))

regr.fit(X_cubic, y_ls)
y_cubic_fit = regr.predict(cubic.fit_transform(X_fit))
r2_cubic = r2_score(y_ls, regr.predict(X_cubic))

plt.figure(figsize=(9, 6))
plt.scatter(X_ls, y_ls, label='training points', color='lightgray')
plt.plot(X_fit, y_lin_fit,   label=f'Linear (d=1),    $R^2$={r2_lin:.3f}',   color='blue',  lw=2, linestyle=':')
plt.plot(X_fit, y_quad_fit,  label=f'Quadratic (d=2), $R^2$={r2_quad:.3f}',  color='red',   lw=2)
plt.plot(X_fit, y_cubic_fit, label=f'Cubic (d=3),     $R^2$={r2_cubic:.3f}', color='green', lw=2, linestyle='--')
plt.xlabel('LSTAT'); plt.ylabel('MEDV')
plt.legend(loc='upper right')
plt.title('직선 vs 2차 vs 3차')
plt.show()

차수를 올릴수록 곡선이 데이터를 더 잘 따라가고, R²도 올라가네요.

**그런데 여기서 멈추면 안 됩니다.** 차수를 계속 올리면 R²는 계속 오르겠지만,
세션자료에서 경고했듯 **과적합**이 생겨요. 학습 데이터만 완벽하게 맞히고 새 데이터에서는 무너지는 상태 말이죠.

그래서 다른 방법을 하나 더 봅시다.

### I-2. 변수를 변환해 보기

산점도를 다시 보면 **우하향하는 지수함수** 모양 같지 않나요?

세션자료의 지수·로그 회귀에서 배운 트릭이 있었죠. **양변에 로그를 취해 직선으로 펴는** 방법이 있었습니다
여기서는 `LSTAT`에 로그를, `MEDV`에 제곱근을 씌워 보겠습니다.

> 💡 로그를 쓸지, 루트를 쓸지는 정해진 공식이 따로 없습니다. **데이터 분포를 보고 경험적으로** 정하거나,
> 몇 가지 해보고 제일 나은 걸 고르는 경우가 많습니다.

In [ ]:
# LSTAT에는 로그를, MEDV에는 제곱근을 씌워 봅시다.
X_log = np.**(X_ls)
y_sqrt = np.**(y_ls)

regr = LinearRegression().fit(X_log, y_sqrt)
r2_trans = r2_score(y_sqrt, regr.predict(X_log))

X_fit_log = np.arange(X_log.min() - 0.1, X_log.max() + 0.1, 0.05)[:, np.newaxis]

plt.figure(figsize=(8, 5))
plt.scatter(X_log, y_sqrt, label='training points', color='lightgray')
plt.plot(X_fit_log, regr.predict(X_fit_log),
         label=f'Linear (d=1), $R^2$={r2_trans:.3f}', color='blue', lw=2)
plt.xlabel('log(LSTAT)'); plt.ylabel('sqrt(MEDV)')
plt.legend(loc='lower left')
plt.title('변환 후 — 직선 하나로 충분해졌습니다')
plt.show()

In [ ]:
summary = pd.DataFrame({
    '모델': ['직선 (d=1)', '2차식 (d=2)', '3차식 (d=3)', 'log/sqrt 변환 후 직선'],
    '항의 개수': [1, 2, 3, 1],
    'R2': [round(r2_lin, 4), round(r2_quad, 4), round(r2_cubic, 4), round(r2_trans, 4)],
})
display(summary)

**Q.** 표를 보고 답해 보세요.

1. 가장 R²가 높은 모델은 무엇인가요?
2. 그 모델은 3차식보다 **복잡한가요, 단순한가요?**
3. 이 결과가 우리에게 알려주는 것은 무엇일까요?

**A.**

<details>
<summary><b>▶ 2번이 헷갈린다면</b></summary>
<br>

3차식은 $x$, $x^2$, $x^3$ 세 개의 항을 씁니다.

변환 모델은 항이 몇 개인가요? **모델의 복잡도**는 항의 개수로 가늠할 수 있어요.

</details>

> 📌 **한 가지를 덧붙이자면,**
>
> 위 R²들은 전부 **학습 데이터 기준**입니다. 진짜로 좋은 모델인지 확인하기 위해서는
> 앞서 했던 것처럼 **테스트 데이터로 다시 확인**해야 해요.
>
> 특히 다항회귀는 차수를 올릴수록 학습 R²가 무조건 올라가기 때문에,
> 테스트 성능을 함께 보지 않으면 과적합을 놓치기 쉽습니다.

---

## 마무리

오늘 실습에서는 다음과 같은 과정을 수행했습니다.

> **직선 긋기 → 잔차로 "왜 이 직선인가" 확인 → 변수 늘리기 → 겹치는 변수 정리
> → 장표로 유의성 확인 → 가정 검정으로 진단 → 성능 평가 → 곡선으로 개선**

그리고 다음 내용들도 한 번씩 더 짚어 봐요!

1. **VIF를 구할 땐 `add_constant()`를 먼저.** 안 그러면 멀쩡한 변수를 지우게 됩니다.
2. **변수는 하나씩 빼면서 확인하기.** 하나를 빼면 나머지 값이 전부 바뀌니까요.
3. **지표마다 답이 다를 수 있습니다.** Adj R², AIC, BIC가 서로 다른 모델을 가리켰죠. 그럴 땐 **분석 목적**을 생각해 보고 지표를 비교해야 합니다.
4. **가정이 깨졌다고 무조건 모델을 버리는 건 아닙니다.** 데이터 크기와 목적을 보고 판단해요.
5. **복잡한 모델이 항상 좋은 것도 아닙니다.** 3차식보다 로그 변환 하나가 더 나았던 것처럼요.

그리고 통계 세션에서 배운 것들이 오늘 계속 등장했다는 점도 눈치채셨을 거예요.
잔차, t-검정, 상관계수와 히트맵, 이상치에 민감한 지표와 둔감한 지표까지요.
**통계를 빠삭하게 이해하고 있어야 회귀의 이해 또한 쉬워진답니다!**

고생 많으셨습니다! 과제에서 만나요 :)